In [26]:
# # Baltimore Marriage Certificates — Docling OCR

# These PDFs are **image-only scans** with blank even pages (backs of scans).

# **Pipeline:**
# 1. Extract odd pages only → save to `data/.../processed/`
# 2. Run Docling OCR on the processed PDF (skips blank pages entirely)

# Docling (Rapid OCR)

In [52]:
from pathlib import Path

import pypdfium2 as pdfium
from docling.datamodel.pipeline_options import PdfPipelineOptions, RapidOcrOptions
from docling.document_converter import DocumentConverter, PdfFormatOption

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data/baltimore-marriage-certificates/raw"
PROCESSED_DIR = PROJECT_ROOT / "data/baltimore-marriage-certificates/processed"

In [53]:
import numpy as np

# A page counts as "blank" (scan backside) if fewer than this fraction of its
# pixels are dark ink. Content pages run ~0.05; true blanks are ~0.00.
BLANK_INK_THRESHOLD = 0.02


def page_ink_ratio(page, dpi: int = 40) -> float:
    """Fraction of dark (ink) pixels on a page, from a fast low-res render."""
    pil = page.render(scale=dpi / 72).to_pil().convert("L")
    return float((np.asarray(pil) < 128).mean())


def detect_content_pages(src_path: Path) -> tuple[list[int], str]:
    """Detect which pages hold certificates (vs blank scan backs).

    These scans alternate content/blank, but the parity differs per file
    (1.pdf -> odd pages, 2.pdf -> even pages). We measure ink per page and
    compare the two parity groups using the *median*, which is robust to the
    occasional dark/noisy blank that would skew a mean.

    Returns:
        (content_indices_0based, human_readable_pattern)
    """
    src = pdfium.PdfDocument(str(src_path))
    try:
        n = len(src)
        ratios = [page_ink_ratio(src[i]) for i in range(n)]
    finally:
        src.close()

    even_group = [ratios[i] for i in range(0, n, 2)]  # pages 1, 3, 5, ...
    odd_group = [ratios[i] for i in range(1, n, 2)]  # pages 2, 4, 6, ...
    even_med = float(np.median(even_group)) if even_group else 0.0
    odd_med = float(np.median(odd_group)) if odd_group else 0.0

    # If neither parity is mostly-blank, there is no alternating pattern:
    # keep every page rather than throwing half away.
    if min(even_med, odd_med) >= BLANK_INK_THRESHOLD:
        return list(range(n)), "all pages (no blank pattern)"

    if even_med >= odd_med:
        return list(range(0, n, 2)), "odd pages (1, 3, 5, ...)"
    return list(range(1, n, 2)), "even pages (2, 4, 6, ...)"


def extract_content_pages(src_path: Path, dst_path: Path) -> dict[int, int]:
    """Copy only the detected certificate pages into a new PDF.

    Returns a mapping: processed_page_number -> original_page_number (1-based).
    """
    content_indices, _pattern = detect_content_pages(src_path)

    src = pdfium.PdfDocument(str(src_path))
    out = pdfium.PdfDocument.new()
    out.import_pages(src, pages=content_indices)

    dst_path.parent.mkdir(parents=True, exist_ok=True)
    out.save(str(dst_path))

    src.close()
    out.close()

    return {new + 1: orig + 1 for new, orig in enumerate(content_indices)}


def process_raw_folder(raw_dir: Path, processed_dir: Path) -> dict[str, dict[int, int]]:
    """Extract certificate pages from every PDF in raw_dir, auto-detecting pattern.

    Returns:
        {raw_filename: page_map} e.g. {"1.pdf": {1: 1, 2: 3, ...}}
    """
    page_maps: dict[str, dict[int, int]] = {}

    for src_path in sorted(raw_dir.glob("*.pdf")):
        content_indices, pattern = detect_content_pages(src_path)
        dst_path = processed_dir / f"{src_path.stem}_content.pdf"
        page_maps[src_path.name] = extract_content_pages(src_path, dst_path)
        print(
            f"{src_path.name}: detected {pattern} -> "
            f"{len(content_indices)} certificates -> {dst_path.name}"
        )

    return page_maps


page_maps = process_raw_folder(RAW_DIR, PROCESSED_DIR)

1.pdf: detected odd pages (1, 3, 5, ...) -> 131 certificates -> 1_content.pdf
2.pdf: detected even pages (2, 4, 6, ...) -> 94 certificates -> 2_content.pdf


In [54]:
processed_pdfs = sorted(PROCESSED_DIR.glob("*_content.pdf"))
print(f"Raw PDFs:       {len(list(RAW_DIR.glob('*.pdf')))} in {RAW_DIR}")
print(f"Processed PDFs: {len(processed_pdfs)} in {PROCESSED_DIR}\n")

for pdf_path in processed_pdfs:
    page_count = len(pdfium.PdfDocument(str(pdf_path)))
    print(f"  {pdf_path.name}: {page_count} pages")

Raw PDFs:       2 in /Users/akhileshmadyastha/Desktop/projects/document-understanding/data/baltimore-marriage-certificates/raw
Processed PDFs: 2 in /Users/akhileshmadyastha/Desktop/projects/document-understanding/data/baltimore-marriage-certificates/processed

  1_content.pdf: 131 pages
  2_content.pdf: 94 pages


In [55]:
pipeline_options = PdfPipelineOptions()

pipeline_options.do_ocr = True
pipeline_options.ocr_options = RapidOcrOptions(
    lang=["english"],
    force_full_page_ocr=True,  # required for image-only scanned PDFs
    backend="onnxruntime",     # avoid broken torch auto-selection when torch is installed
)

In [56]:
converter = DocumentConverter(
    format_options={
        "pdf": PdfFormatOption(pipeline_options=pipeline_options),
    }
)

In [57]:
results: dict[str, object] = {}

for result in converter.convert_all(processed_pdfs):
    filename = Path(result.input.file).name
    results[filename] = result
    print(f"Converted {filename}: {len(result.document.pages)} pages")

print(f"\nTotal converted: {len(results)} PDFs")

[INFO] 2026-07-03 12:10:30,142 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-07-03 12:10:30,154 [RapidOCR] download_file.py:60: File exists and is valid: /Users/akhileshmadyastha/Desktop/projects/document-understanding/.venv/lib/python3.11/site-packages/rapidocr/models/PP-OCRv6_det_small.onnx
[INFO] 2026-07-03 12:10:30,155 [RapidOCR] main.py:63: Using /Users/akhileshmadyastha/Desktop/projects/document-understanding/.venv/lib/python3.11/site-packages/rapidocr/models/PP-OCRv6_det_small.onnx
[INFO] 2026-07-03 12:10:30,244 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-07-03 12:10:30,247 [RapidOCR] download_file.py:60: File exists and is valid: /Users/akhileshmadyastha/Desktop/projects/document-understanding/.venv/lib/python3.11/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-07-03 12:10:30,247 [RapidOCR] main.py:63: Using /Users/akhileshmadyastha/Desktop/projects/document-understanding/.venv/lib/python3.11/site-packages

Converted 1_content.pdf: 131 pages
Converted 2_content.pdf: 94 pages

Total converted: 2 PDFs


In [59]:
# Inspect conversion results

def summarize_conversion_results(results: dict[str, object]) -> None:
    """Print a quick overview of every converted PDF."""
    if not results:
        print("No results yet. Run the conversion cell first.")
        return

    print(f"{'File':<22} {'Status':<26} {'Pages':>5}  {'Errors':>6}")
    print("-" * 65)
    for filename, result in sorted(results.items()):
        n_errors = len(result.errors or [])
        print(
            f"{filename:<22} {str(result.status):<26} "
            f"{len(result.document.pages):>5}  {n_errors:>6}"
        )
    print(f"\nTotal converted: {len(results)} PDF(s)")


def show_page_text(results: dict[str, object], filename: str, page_no: int = 1) -> str:
    """Return OCR text for one page of a converted PDF."""
    doc = results[filename].document
    return doc.export_to_text(page_no=page_no)


summarize_conversion_results(results)

print("\n--- Page 1 of 1_content.pdf ---\n")
print(show_page_text(results, "1_content.pdf", page_no=1))

File                   Status                     Pages  Errors
-----------------------------------------------------------------
1_content.pdf          ConversionStatus.SUCCESS     131       0
2_content.pdf          ConversionStatus.SUCCESS      94       0

Total converted: 2 PDF(s)

--- Page 1 of 1_content.pdf ---

GOOD ONLY IN-BALTIMORE CITY

Marriage License STATE OF MARYLAND

BALTIMORE CITY

No. 34178

APPLI  5221

IHereby Certify, That on this Jo day ot Aug

Nineteen hundred and thirty

in accordance with the License issued by the

Clerk of the Court of Common Pleas herein indicated.

JAMES S CRAWFORD

and ALLIME POYAS

(Name of troom)

(Name of Bride)

Residence of Groom BALTO MD

Age 29 Color COLNativity TENor country)

Occupation MUSICIAN

and who is.

(State here widower or divorced as the case may be)

Residence of BrideDALTO D

(State or Country)

OccupationONE

and who is.

Name of Person consenting if Groom is a Minor.

(Parent or Guardian)

Name of Person consenting if B

In [60]:
result.status

<ConversionStatus.SUCCESS: 'success'>

In [61]:
result.errors 

[]

In [62]:
doc = result.document

# Plain text of one page (what you saw for the marriage license)
print(doc.export_to_text(page_no=1))

GOOD ONLY IN BALTIMORE CITY

No. 36829

Marriage License

STATE OFMARYLAND

BALTIMORE CITY

APP 11064

Die.

FHereby Certity, That on this

Nineteen hundred and thirty

Balte.

the below, were by me united in marriage at,

in accordance with the License issued by the

Clerk of the Court of Common Pleas hereln indicated.

JOHN J. GUTHRIE

DOROTHY WHITFIELD

and

(Name of Groom)

(Name of Bride)

BATIMORE MD.

23

COL

NEW YORK

Residence of Groom.

Age

Color

Nativity

(State or Country)

CHAFFEUR

SINGLE

Occupation.

and who is.

(State here whether single, widower or divorced as the case may be)

BALTIMORE MD.

20

COL

MAFYLAND

Residence of Bride.

Age.

Color.

Nativity.

MAID

SINGLE

(State or Country)

Occupation

and who is.

(State bere whetber single, widow or divorced as the case may be)

Name of Person consenting if Groom is a Minor.

(Parent or Guardian)

Name of Person consenting if Bride is a Minor.

(Parent or Guardian)

NOT RELATED

Y.IM. OAarris Signature of Officia

# Saving Results

In [63]:
from pathlib import Path
from docling.datamodel.document import ConversionAssets
from docling_core.types.doc import DoclingDocument

In [64]:

DOCLING_DIR = PROJECT_ROOT / "artifacts/docling"
DOCLING_DIR.mkdir(parents=True, exist_ok=True)


# Save after conversion
for filename, result in results.items():
    stem = Path(filename).stem
    # Full result (status, timings, confidence, document) → .zip
    result.save(filename=DOCLING_DIR / f"{stem}.docling.zip")
    # Document only → .json (lighter, easier to inspect)
    result.document.save_as_json(DOCLING_DIR / f"{stem}.json")
    print(f"Saved {stem}")

Saved 1_content
Saved 2_content


In [65]:
# Option A: full saved result (use ConversionAssets, not ConversionResult)

loaded = ConversionAssets.load(DOCLING_DIR / "1_content.docling.zip")
print(loaded.document.export_to_text(page_no=1))



# # Option B: document JSON only
# doc = DoclingDocument.load_from_json(DOCLING_DIR / "1_content.json")
# print(doc.export_to_text(page_no=1))

GOOD ONLY IN-BALTIMORE CITY

Marriage License STATE OF MARYLAND

BALTIMORE CITY

No. 34178

APPLI  5221

IHereby Certify, That on this Jo day ot Aug

Nineteen hundred and thirty

in accordance with the License issued by the

Clerk of the Court of Common Pleas herein indicated.

JAMES S CRAWFORD

and ALLIME POYAS

(Name of troom)

(Name of Bride)

Residence of Groom BALTO MD

Age 29 Color COLNativity TENor country)

Occupation MUSICIAN

and who is.

(State here widower or divorced as the case may be)

Residence of BrideDALTO D

(State or Country)

OccupationONE

and who is.

Name of Person consenting if Groom is a Minor.

(Parent or Guardian)

Name of Person consenting if Bride is a Minor

(Parent or Guardian)

(Stae   t e e e e e  ted)

J. M. Harris

Signature of Officiating Clergyman

Official Character

Address

This Certificate to be retained by the office of the Clerk of the Court of Common Pleas for above named city.
